## Transform US Census GeoSpatial data on Hugging Face to parquet using DuckDB



In [1]:
import duckdb


In [2]:
# Connect to DuckDB and load the spatial extension
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;") ;


In [3]:
roads_url = "https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_35006_roads.zip"
states_url = "https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/tl_2024_us_state.zip"


In [6]:
# Extract the base file name
roads_base = roads_url.split("/")[-1].split(".")[0]
states_base = states_url.split("/")[-1].split(".")[0]

roads_base, states_base


('tl_2024_35006_roads', 'tl_2024_us_state')

In [28]:
! curl -L -O {roads_url}
! curl -L -O {states_url}


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1002  100  1002    0     0   5103      0 --:--:-- --:--:-- --:--:--  5138
100 2368k  100 2368k    0     0  1249k      0  0:00:01  0:00:01 --:--:-- 3753k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1000  100  1000    0     0   6717      0 --:--:-- --:--:-- --:--:--  6756
100 9721k  100 9721k    0     0  4886k      0  0:00:01  0:00:01 --:--:-- 12.0M


In [29]:
! ls -la {roads_base}* {states_base}*


-rw-r--r-- 1 root root 2425481 Nov 18 15:19 tl_2024_35006_roads.zip
-rw-r--r-- 1 root root 9954307 Nov 18 15:19 tl_2024_us_state.zip


In [30]:
roads_path = f"/vsizip/{roads_base}.zip"
states_path = f"/vsizip/{states_base}.zip"

roads_path, states_path


('/vsizip/tl_2024_35006_roads.zip', '/vsizip/tl_2024_us_state.zip')

## Convert to Parquet


In [32]:
# Export Roads shapefile to GeoParquet
con.execute(f"""
    COPY (
        SELECT * FROM st_read('{roads_path}')
    ) TO '{roads_base}.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
""")



In [33]:
# Export States shapefile to GeoParquet
con.execute(f"""
    COPY (
        SELECT * FROM st_read("{states_path}")
    ) TO '{states_base}.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
""")


In [36]:
query = f"""
    SELECT
        roads.* EXCLUDE (geom),
        states.STATEFP,
        states.STUSPS,
        states.NAME AS state_name,
        roads.geom AS road_geometry,
        states.geom AS state_geometry
    FROM read_parquet('{roads_base}.parquet') roads
    LEFT JOIN read_parquet('{states_base}.parquet') states
        ON ST_Within(roads.geom, states.geom)
    WHERE roads.FULLNAME LIKE '%I- 25%'
       OR (roads.RTTYP = 'I'
       -- AND roads.FULLNAME LIKE '%25%'
       )
"""
print(query)



    SELECT
        roads.* EXCLUDE (geom),
        states.STATEFP,
        states.STUSPS,
        states.NAME AS state_name,
        roads.geom AS road_geometry,
        states.geom AS state_geometry
    FROM read_parquet('tl_2024_35006_roads.parquet') roads
    LEFT JOIN read_parquet('tl_2024_us_state.parquet') states
        ON ST_Within(roads.geom, states.geom)
    WHERE roads.FULLNAME LIKE '%I- 25%'
       OR (roads.RTTYP = 'I'
       -- AND roads.FULLNAME LIKE '%25%'
       )



In [37]:
gdf_combined = con.sql(query).df()
gdf_combined


,LINEARID,FULLNAME,RTTYP,MTFCC,STATEFP,STUSPS,state_name,road_geometry,state_geometry
0,1104492838447,I- 40,I,S1100,35,NM,New Mexico,"[1, 4, 0, 0, 0, 0, 0, 0, 52, 251, 215, 194, 68...","[2, 4, 0, 0, 0, 0, 0, 0, 211, 25, 218, 194, 67..."
1,1104492838321,I- 40,I,S1100,35,NM,New Mexico,"[1, 4, 0, 0, 0, 0, 0, 0, 4, 251, 215, 194, 129...","[2, 4, 0, 0, 0, 0, 0, 0, 211, 25, 218, 194, 67..."


## Push to Hugging Face
